In [ ]:
from pathlib import Path
import random
import numpy as np
import torch

from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.sasrec import SASRecConfig, SASRec
from flowrec.ft_methods.sft import SFTConfig, SFTMethod
from flowrec.training import TrainerConfig, Trainer

from flowrec.ft_methods.sft.metrics import item_ranking_metrics

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
@torch.no_grad()
def compute_test_hit_ndcg_coverage(
    model: torch.nn.Module,
    dm: AmazonReviewDataModule,
    *,
    ks=(5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    """Compute NDCG@K, HitRate@K, Coverage@K on test data
    """
    device = next(model.parameters()).device
    model.eval()

    vocab_size = dm.item_vocab_size
    recommendable = max(vocab_size - 1, 1)

    # Accumulate instance-level hit and ndcg
    hits = {k: [] for k in ks}
    ndcg_sums = {k: 0.0 for k in ks}
    ndcg_counts = {k: 0 for k in ks}

    recommended_sets = {k: set() for k in ks}

    for batch in dm.test_dataloader():
        for key, val in batch.items():
            if torch.is_tensor(val):
                batch[key] = val.to(device, non_blocking=True)

        out = model(batch)
        scores = out.scores  # [B, V]
        target_item_ids = batch["target_item_ids"]

        targets = target_item_ids.long()
        valid = targets != pad_item_id
        if valid.sum() == 0:
            continue

        max_k = max(ks)
        topk_items = torch.topk(scores, k=max_k, dim=-1).indices  # [B, max_k]
        topk_valid = topk_items[valid]
        targets_valid = targets[valid]

        for k in ks:
            kk = int(k)
            topk_k = topk_valid[:, :kk]

            matches = topk_k.eq(targets_valid.unsqueeze(1))
            hit = matches.any(dim=1)
            hits[k].append(hit.float().cpu())

            rank_pos = matches.float().argmax(dim=1)
            ndcg = torch.zeros_like(hit, dtype=torch.float32)
            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank_pos[hit].float() + 2.0)
            ndcg_sums[k] += float(ndcg.mean().item())
            ndcg_counts[k] += 1

            uniq = torch.unique(topk_k).detach().cpu().tolist()
            recommended_sets[k].update(int(x) for x in uniq if int(x) != pad_item_id)

    metrics: dict[str, float] = {}
    for k in ks:
        hit_t = torch.cat(hits[k], dim=0) if hits[k] else torch.zeros(0)
        hit_rate = float(hit_t.mean().item()) if hit_t.numel() else 0.0
        coverage = len(recommended_sets[k]) / recommendable
        ndcg = ndcg_sums[k] / max(ndcg_counts[k], 1)
        metrics[f"ndcg@{k}"] = ndcg
        metrics[f"hit_rate@{k}"] = hit_rate
        metrics[f"coverage@{k}"] = coverage

    return metrics

In [ ]:
reviews_path = Path("../data/raw/reviews_Beauty.json.gz")
metadata_path = Path("../data/raw/meta_Beauty.json.gz")

data_cfg = AmazonReviewDataConfig(
    reviews_path=reviews_path,
    metadata_path=metadata_path,
    semantic_id_path=None,
    min_user_interactions=5,
    min_item_interactions=5,
    split_strategy="leave_one_out",
    max_history_len=50,
    history_padding="right",
    batch_size=256,
    eval_batch_size=512,
)

dm = AmazonReviewDataModule(data_cfg)
dm.prepare_data()
dm.setup()

print("Dataset summary:")
print(dm.summary())
print("Item vocab size:", dm.item_vocab_size)

In [ ]:
model_cfg = SASRecConfig(
    item_vocab_size=dm.item_vocab_size,
    max_history_len=data_cfg.max_history_len,
    d_model=256,
    nhead=8,
    num_layers=3,
    dropout=0.1,
)

sft_cfg = SFTConfig()
method_cfg = sft_cfg

trainer_cfg = TrainerConfig(
    max_epochs=10,
    gradient_accumulation_steps=4,
    use_amp=True,
    log_every_steps=50,
    validate_every_epochs=2,
    monitor="val/loss",
    early_stopping_patience=3,
    checkpoint_dir="../artifacts/sasrec_beauty",
)

In [ ]:
seeds = [42, 43, 44]
ks = (5, 10, 20)
all_seed_metrics = []

for seed in seeds:
    print(f"\n===== Train/eval seed={seed} =====")
    set_seed(seed)

    model = SASRec(model_cfg)
    method = SFTMethod(method_cfg)

    trainer = Trainer.from_optimizer_config(
        model=model,
        method=method,
        cfg=trainer_cfg,
        lr=3e-4,
        weight_decay=1e-2,
    )

    _ = trainer.fit(dm.train_dataloader(), dm.val_dataloader())

    ckpt_path = f"../artifacts/sasrec_beauty/final_sasrec_seed_{seed}.pt"
    model.save_checkpoint(ckpt_path)

    test_metrics = compute_test_hit_ndcg_coverage(model, dm, ks=ks)
    test_metrics["seed"] = seed

    print("Test metrics:")
    for key in sorted([k for k in test_metrics.keys() if k != 'seed']):
        print(f"  {key}: {test_metrics[key]:.6f}")

    all_seed_metrics.append(test_metrics)

In [ ]:
df.to_csv("metrics_10_epochs.csv", index=False)

In [ ]:
df

In [ ]:
df_mean_std = pd.DataFrame({
    col: [f"{df[col].mean():.3f} ± {3 * df[col].std():.3f}"]
    for col in df.select_dtypes(include='number').columns
})
df_mean_std = df_mean_std.drop(columns=['seed'])
df_mean_std

In [ ]:
df.to_csv("data.csv", index=False)

In [ ]:
trainer.state